# Module 4 — Grouping, lagging and merging


Most of the second half of the course is three moves: **group** rows that belong
together, **lag** a column so a row can see an earlier one, and **merge** two tables that
describe the same thing.

The files: `fin400-stock-panel.csv` is one row per firm-year (`ticker`, `fyear`, `ni`,
`debt_lgterm`, `mcap`, `shr`); `fin400-monthly-stocks.csv` is one row per stock-month
(`permno`, `caldt`, `prc`, `ret`, `mcap`); `fin400-firms.csv` maps `permno` to a ticker
and sector.


## 1. Group by a column, then aggregate


In [1]:
import pandas as pd

panel = pd.read_csv("fin400-stock-panel.csv")
panel["levratio"] = panel["debt_lgterm"] / (panel["debt_lgterm"] + panel["mcap"])

print(panel["levratio"].median())
print(panel.groupby("fyear")["levratio"].median().head())


0.3694843824659694
fyear
1980    0.645962
1981    0.366122
1982    0.366558
1983    0.408615
1984    0.394853
Name: levratio, dtype: float64


`groupby("fyear")` splits the rows into one pile per year, and the aggregate collapses
each pile to a single number. "The typical firm" means the **median**, not the mean.


## 2. Merge that summary back onto every row


In [2]:
by_year = panel.groupby("fyear")["levratio"].median().reset_index()
by_year = by_year.rename(columns={"levratio": "levratio_median_year"})

panel = pd.merge(panel, by_year, on="fyear", how="left")
print(panel[["ticker", "fyear", "levratio", "levratio_median_year"]].head())


  ticker  fyear  levratio  levratio_median_year
0   BJYB   1987  0.380609              0.395458
1   BJYB   1988  0.289306              0.331179
2   BJYB   1989  0.417384              0.366830
3   BJYB   1990  0.254937              0.351986
4   BJYB   1991  0.354206              0.371849


`reset_index()` is what turns the year index back into a column that can be merged. A
`left` merge keeps every firm-year and attaches the year's median; because the right
table has one row per year, no row is duplicated.


## 3. Lag a column inside each stock


In [3]:
stocks = pd.read_csv("fin400-monthly-stocks.csv", parse_dates=["caldt"])
stocks = stocks.sort_values(["permno", "caldt"])

stocks["mcap_lag"] = stocks.groupby("permno")["mcap"].shift(1)

print(stocks[["permno", "caldt", "mcap", "mcap_lag"]].head())
print(int(stocks["mcap_lag"].isna().sum()))


     permno      caldt     mcap  mcap_lag
0     10001 1926-01-01  4449.11       NaN
45    10001 1926-02-01  4888.51   4449.11
90    10001 1926-03-01  4682.26   4888.51
135   10001 1926-04-01  3702.73   4682.26
180   10001 1926-05-01  3136.49   3702.73
45


Sort first, then `shift(1)` inside each `permno`: the first month of every stock has no
previous month, so it becomes `NaN`. Skipping the sort lets one company inherit another
company's values.


## 4. Weighted returns per month


In [4]:
stocks["weighted"] = stocks["ret"] * stocks["mcap"]
vw = stocks.groupby("caldt")["weighted"].sum() / stocks.groupby("caldt")["mcap"].sum()

print(round(float(vw.mean()), 4))
print(vw.head())


0.0164
caldt
1926-01-01    0.022638
1926-02-01    0.141458
1926-03-01    0.028797
1926-04-01   -0.028913
1926-05-01    0.028670
dtype: float64


An equal-weight portfolio would use `groupby("caldt")["ret"].mean()` instead. The
difference between the two is the size effect — small stocks get equal say in one and
almost none in the other.


## 5. Merge two tables


In [5]:
size = pd.read_csv("fin400-mcap-emp.csv")
firms = pd.read_csv("fin400-firms.csv")

merged = pd.merge(size, firms, on="permno", how="inner")
print(merged.shape)
print(list(merged.columns))
print(merged.groupby("sector")["mcap"].mean().round(1))


(600, 6)
['permno', 'fyear', 'mcap', 'emp', 'ticker', 'sector']
sector
Consumer       1861.0
Energy         1958.0
Financials     1760.4
Health Care    1534.7
Industrials    1841.6
Technology     2046.6
Name: mcap, dtype: float64


## Try it

1. Compute the median leverage ratio by year and find the year with the highest value.
2. Count the stocks in each month, then explain why `mcap_lag` has fewer non-missing
   values than `mcap`.
3. Merge the company file onto the panel by ticker and report mean market cap by sector.

**Practice 10** to **Practice 16** on the practice page are built from these files.
